In [1]:
!pip install wandb gensim sentence-transformers transformers sentencepiece faiss-cpu peft -q
!pip uninstall -y hf_xet -q

import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import os
os.environ["WANDB_SILENT"] = "true"

from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
w_key = user_secrets.get_secret("WANDB_Key")
wandb.login(key=w_key)

import torch
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration, pipeline, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, TaskType
import time

def load_with_retry(load_fn, *args, retries=5, delay=8, **kwargs):
    for attempt in range(retries):
        try:
            return load_fn(*args, **kwargs)
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"download failed (attempt {attempt+1}/{retries}), retrying in {delay}s...")
            time.sleep(delay)
OPTS   = ['A', 'B', 'C', 'D', 'E']
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| device:", device)
train.head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.8 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [2]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

=== null counts ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== answer distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

=== prompt length (words) ===
count    2000.00
mean       18.15
std         6.78
min         3.00
25%        14.00
50%        17.00
75%        22.00
max        51.00
Name: prompt, dtype: float64

=== avg option length (words) ===
  A: 26.1
  B: 26.5
  C: 26.5
  D: 26.0
  E: 26.2


In [3]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

unique vocab in cleaned prompts: 859
sample — original : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
sample — cleaned  : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options


In [4]:
from datasets import Dataset, Dataset as MCQDataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

hf_train = Dataset.from_pandas(train)
hf_test  = Dataset.from_pandas(test)
print(hf_train)

# tfidf on full corpus so test vocab is covered (no pretrained weights)
full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=25000, sublinear_tf=True)
tfidf.fit(full_corpus)
print(f"tfidf vocab: {len(tfidf.vocabulary_)}")

# word2vec on cleaned stopwordremoved tokens from scratch
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))
w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
print(f"w2v vocab: {len(w2v_model.wv)}")

# roberta pretrained model used both for cosine sim baseline and attention inspection
bert_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
bert_model     = AutoModel.from_pretrained("roberta-base", output_attentions=True).to(device)
bert_model.eval()
sample_inputs = bert_tokenizer(train['prompt'].iloc[0], return_tensors="pt").to(device)
with torch.no_grad():
    sample_out = bert_model(**sample_inputs)
print(f"\nattention layers: {len(sample_out.attentions)}, heads per layer: {sample_out.attentions[0].shape[1]}")
print("this is how roberta learns which words to focus on relative to others")

# data formatting for mcq
# concatenating question with all 5 options into a single input string, model predicts the correct letter
LETTER2ID = {o: i for i, o in enumerate(OPTS)}
ID2LETTER = {i: o for o, i in LETTER2ID.items()}

def format_mcq(row):
    return (f"{row['prompt']} "
            f"A) {row['A']} B) {row['B']} C) {row['C']} D) {row['D']} E) {row['E']}")

train_inputs = [format_mcq(row) for _, row in train.iterrows()]
train_labels = [LETTER2ID[a] for a in train['answer']]
print(f"\nsample formatted input: {train_inputs[0][:150]}...")

# reusing bert_tokenizer from roberta-base,
cls_tok  = bert_tokenizer
cls_base = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5)

cls_enc = cls_tok(train_inputs, truncation=True, max_length=256, padding=False)
cls_enc['labels'] = train_labels
hf_cls  = MCQDataset.from_dict(cls_enc)
print(f"training examples: {len(hf_cls)}")

# lora freezes the base model and only trains small rank decomposition matrices added to attention q and v
# this cuts trainable params and gpu memory drastically while reaching similar accuracy
full_params = sum(p.numel() for p in cls_base.parameters())
print(f"full model params (if full finetuning): {full_params:,}")

lora_cfg = LoraConfig(
    task_type      = TaskType.SEQ_CLS,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none"
)

lora_model = get_peft_model(cls_base, lora_cfg).to(device)
lora_model.print_trainable_parameters()

cls_collator = DataCollatorWithPadding(cls_tok)

# fp16 halves memory per tensor so bigger batch can be used
# gradient accumulationsteps simulates a larger effective batch without needing more vram
cls_args = TrainingArguments(
    output_dir                  = "/kaggle/working/lora_cls",
    num_train_epochs            = 6,
    per_device_train_batch_size = 16,
    gradient_accumulation_steps = 2,
    learning_rate               = 2e-4,
    warmup_steps                = 50,
    fp16                        = True,
    logging_steps               = 50,
    save_strategy               = "no",
    report_to                   = "none",
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0
)

cls_trainer = Trainer(
    model            = lora_model,
    args             = cls_args,
    train_dataset    = hf_cls,
    data_collator    = cls_collator,
    processing_class = cls_tok
)

print("starting lora classification finetuning...")
cls_trainer.train()
lora_model.eval()
print("lora finetuning done")

# sanity check predictions should vary across rows, not stuck on one letter
print("\nsanity check on first 5 train rows:")
for i in range(5):
    row_i  = train.iloc[i]
    text_i = format_mcq(row_i)
    ins_i  = cls_tok(text_i, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits_i = lora_model(**ins_i).logits[0]
    pred_i = ID2LETTER[int(torch.argmax(logits_i))]
    print(f"row {i} | predicted: '{pred_i}' | actual: {row_i['answer']}")

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'clean_prompt', 'clean_A', 'clean_B', 'clean_C', 'clean_D', 'clean_E'],
    num_rows: 2000
})
tfidf vocab: 10915


w2v vocab: 2883


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


attention layers: 12, heads per layer: 12
this is how roberta learns which words to focus on relative to others

sample formatted input: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. A) Mart...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

training examples: 2000
full model params (if full finetuning): 124,649,477
trainable params: 1,184,261 || all params: 125,833,738 || trainable%: 0.9411
starting lora classification finetuning...
{'loss': '6.343', 'grad_norm': '7.237', 'learning_rate': '0.000196', 'epoch': '1.571'}
{'loss': '5.098', 'grad_norm': '16.92', 'learning_rate': '0.000131', 'epoch': '3.127'}
{'loss': '2.561', 'grad_norm': '11.53', 'learning_rate': '6.056e-05', 'epoch': '4.698'}
{'train_runtime': '231.9', 'train_samples_per_second': '51.74', 'train_steps_per_second': '0.828', 'train_loss': '3.944', 'epoch': '6'}
lora finetuning done

sanity check on first 5 train rows:
row 0 | predicted: 'B' | actual: B
row 1 | predicted: 'A' | actual: A
row 2 | predicted: 'C' | actual: C
row 3 | predicted: 'B' | actual: B
row 4 | predicted: 'A' | actual: A


In [5]:
def tfidf_cosine(text1, text2):
    v1 = tfidf.transform([text1])
    v2 = tfidf.transform([text2])
    return float(cosine_similarity(v1, v2)[0][0])

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1)
    v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

# roberta mean pooling ignores padding tokens via attention mask
def get_roberta_embedding(text):
    inputs     = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
    with torch.no_grad():
        out    = bert_model(**inputs)
    token_embs = out.last_hidden_state
    mask       = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
    summed     = torch.sum(token_embs * mask, dim=1)
    counts     = torch.clamp(mask.sum(dim=1), min=1e-9)
    return (summed / counts).squeeze(0).cpu().numpy()

def rank_tfidf(row):
    sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_roberta(row):
    pv   = get_roberta_embedding(row['prompt']).reshape(1, -1)
    sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

# lora classification ranking, logits give score per option directly
def rank_lora(row):
    text   = format_mcq(row)
    inputs = cls_tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = lora_model(**inputs).logits[0]
    scores = {ID2LETTER[i]: logits[i].item() for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

from sklearn.metrics import accuracy_score, f1_score

# accuracy and f1 computed on top-1 prediction 
# separate from mAP@3 which measures ranking quality across top-3
def top1_accuracy(actuals, predictions):
    top1_preds = [p[0] for p in predictions]
    return round(accuracy_score(actuals, top1_preds), 4)

def top1_f1(actuals, predictions):
    top1_preds = [p[0] for p in predictions]
    return round(f1_score(actuals, top1_preds, average='macro'), 4)

actuals        = train['answer'].tolist()
top3_maj       = list(train['answer'].value_counts().index[:3])
map3_maj       = round(mapk(actuals, [top3_maj] * len(train)), 4)
tfidf_preds    = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
map3_tfidf     = round(mapk(actuals, tfidf_preds), 4)
acc_tfidf = top1_accuracy(actuals, tfidf_preds)
f1_tfidf  = top1_f1(actuals, tfidf_preds)

w2v_preds      = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v       = round(mapk(actuals, w2v_preds), 4)
acc_w2v = top1_accuracy(actuals, w2v_preds)
f1_w2v  = top1_f1(actuals, w2v_preds)

sample_roberta = train.sample(n=200, random_state=42)
roberta_preds  = [rank_roberta(row)[:3] for _, row in sample_roberta.iterrows()]
map3_roberta   = round(mapk(sample_roberta['answer'].tolist(), roberta_preds), 4)
acc_roberta = top1_accuracy(sample_roberta['answer'].tolist(), roberta_preds)
f1_roberta  = top1_f1(sample_roberta['answer'].tolist(), roberta_preds)

print("evaluating lora on full train...")
lora_preds = [rank_lora(row)[:3] for _, row in train.iterrows()]
map3_lora  = round(mapk(actuals, lora_preds), 4)
acc_lora = top1_accuracy(actuals, lora_preds)
f1_lora  = top1_f1(actuals, lora_preds)

print(f"\nmajority baseline    MAP@3 : {map3_maj}")
print(f"tfidf cosine         MAP@3 : {map3_tfidf}")
print(f"word2vec cosine      MAP@3 : {map3_w2v}")
print(f"roberta cosine (200) MAP@3 : {map3_roberta}")
print(f"lora finetuned       MAP@3 : {map3_lora}")

print(f"\ntfidf   — acc: {acc_tfidf}, f1: {f1_tfidf}")
print(f"w2v     — acc: {acc_w2v}, f1: {f1_w2v}")
print(f"roberta — acc: {acc_roberta}, f1: {f1_roberta}")
print(f"lora    — acc: {acc_lora}, f1: {f1_lora}")

run = wandb.init(
    project = "23f1000362-t22026",
    name    = "milestone4-final-4models",
    config  = {
        "milestone": 4, "lora_r": 16, "lora_alpha": 32,
        "epochs": 6, "lr": 2e-4, "fp16": True, "base_model": "roberta-base"
    },
    tags = ["milestone-4", "lora", "final"]
)
wandb.log({
    "map3/majority_baseline"  : map3_maj,
    "map3/tfidf_cosine"       : map3_tfidf,
    "map3/word2vec_cosine"    : map3_w2v,
    "map3/roberta_cosine_200" : map3_roberta,
    "map3/lora_finetuned"     : map3_lora,
    "accuracy/tfidf"          : acc_tfidf,
    "accuracy/word2vec"       : acc_w2v,
    "accuracy/roberta"        : acc_roberta,
    "accuracy/lora"           : acc_lora,
    "f1/tfidf"                : f1_tfidf,
    "f1/word2vec"             : f1_w2v,
    "f1/roberta"              : f1_roberta,
    "f1/lora"                 : f1_lora
})
wandb.finish()


test_preds = [" ".join(rank_lora(row)[:3]) for _, row in test.iterrows()] # lora as submission model coz highest score, 
submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
submission.head(5)

evaluating lora on full train...

majority baseline    MAP@3 : 0.4213
tfidf cosine         MAP@3 : 0.3228
word2vec cosine      MAP@3 : 0.2918
roberta cosine (200) MAP@3 : 0.3425
lora finetuned       MAP@3 : 0.9874

tfidf   — acc: 0.1655, f1: 0.1616
w2v     — acc: 0.147, f1: 0.1456
roberta — acc: 0.185, f1: 0.1819
lora    — acc: 0.977, f1: 0.9762
submission.csv saved


,ID,Prediction
0,1,A D C
1,2,B E C
2,3,B D A
3,4,E C B
4,5,C E A
